# Importando bibliotecas...

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from functools import reduce

# Lendo tabela da camada bronze

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

In [0]:
df.limit(10).display()

In [0]:
%sql
select * 
from workspace.silver.erp_customers
limit 10

# Limpeza de espaços em branco com Trimming

In [0]:
for i in df.schema.fields:
    if isinstance(i.dataType, StringType):
        df = df.withColumn(i.name, F.trim(F.col(i.name)))

# Limpeza do campo CID

In [0]:
df = df.withColumn('CID', F.regexp_replace(F.col('CID'), '-', ''))#.limit(10).display()

# Normalização da coluna country

In [0]:
df.select('CNTRY').distinct().display()

In [0]:
df = df.withColumn(
    'CNTRY',
    F.when(F.col('CNTRY') == 'DE', 'Germany')
     .when(F.col('CNTRY').isin('US', 'USA'), 'United States')
     .when((F.col('CNTRY') == '') | F.col('CNTRY').isNull(), 'n/a')
     .otherwise(F.col('CNTRY'))
)#.select('CNTRY').distinct().display()

# Renomeando colunas

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Escrita na camada Prata

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customer_location')

# Verificação da tabela

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_customer_location
LIMIT 10